In [74]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score,mean_squared_error
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

In [52]:
df=pd.read_csv('/content/drive/MyDrive/gradtwin project/traffic-forecasting-system-main/Metro_Interstate_Traffic_Volume.csv')
df

,Temp,Rain_1H,Snow_1H,Clouds_All,Weather_Main,Weather_Description,Traffic_Volume,Date,Time
0,288.28,0.0,0.0,40,Clouds,scattered clouds,5545,2012-10-02,09:00:00
1,289.36,0.0,0.0,75,Clouds,broken clouds,4516,2012-10-02,10:00:00
2,289.58,0.0,0.0,90,Clouds,overcast clouds,4767,2012-10-02,11:00:00
3,290.13,0.0,0.0,90,Clouds,overcast clouds,5026,2012-10-02,12:00:00
4,291.14,0.0,0.0,75,Clouds,broken clouds,4918,2012-10-02,13:00:00
...,...,...,...,...,...,...,...,...,...
48199,283.45,0.0,0.0,75,Clouds,broken clouds,3543,2018-09-30,19:00:00
48200,282.76,0.0,0.0,90,Clouds,overcast clouds,2781,2018-09-30,20:00:00
48201,282.73,0.0,0.0,90,Thunderstorm,proximity thunderstorm,2159,2018-09-30,21:00:00
48202,282.09,0.0,0.0,90,Clouds,overcast clouds,1450,2018-09-30,22:00:00


In [55]:
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])

In [56]:
# Extract numerical features from time
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month

In [57]:
# Encode categorical columns (Weather_Main and Weather_Description)
le = LabelEncoder()
df['Weather_Main'] = le.fit_transform(df['Weather_Main'])
df['Weather_Description'] = le.fit_transform(df['Weather_Description'])

In [58]:
X = df.drop(['Traffic_Volume', 'Date', 'Time', 'datetime'], axis=1)
y = df['Traffic_Volume']

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [68]:
rf=RandomForestRegressor()

In [69]:
param_dist = {
'n_estimators': [50,100,200,300,400],
'max_depth': [None,10,20,30],
'min_samples_split': [2,5,10],
'min_samples_leaf': [1,2,4],'max_features': ['auto', 'sqrt', 'log2'],
'bootstrap': [True, False]
}

In [70]:
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

In [71]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


RandomizedSearchCV(cv=3, estimator=RandomForestRegressor(), n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [None, 10, 20, 30],
                                        'max_features': ['auto', 'sqrt',
                                                         'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [50, 100, 200, 300,
                                                         400]},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [72]:
# The parameters that got us there
print(f"Best Parameters: {random_search.best_params_}")

Best Parameters: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30, 'bootstrap': False}


In [73]:
best_model = random_search.best_estimator_
predictions = best_model.predict(X_test)

In [75]:
print(f"\nR2 Score: {r2_score(y_test, predictions):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.2f}")


R2 Score: 0.9566
RMSE: 414.05


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
